# Limpieza y modelo de datos
**Objetivo:** aplicar las tareas de limpieza definidas en el paso 9 de `1_eda.ipynb` y guardar las tablas limpias en `datos/procesados`.

**Criterio general:** no se elimina ninguna fila ni se inventan datos. Las filas que no son países (agregados) se separan en una tabla aparte, las filas problemáticas se marcan con banderas y cada indicador se calcula solo con las filas válidas para él. Los archivos de `datos/raw` no se modifican.

**Entradas** (`datos/raw`): Comext importaciones de España y de la UE, Datacomex por país de origen y TIO 2022.
**Salidas** (`datos/procesados`): tablas limpias de Comext, Datacomex y TIO.

In [1]:
# Librerías y formato de números (inglés, 2 decimales), igual que en el EDA
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.float_format", "{:,.2f}".format)

RAW = Path("datos/raw")
PROC = Path("datos/procesados")
PROC.mkdir(exist_ok=True)

## ✅Paso 0. Carga de los datos
**Enunciado:** Leer los ficheros de `datos/raw` como texto, igual que en la exploración, para no alterar códigos ni decimales.

In [2]:
# Comext (Eurostat): importaciones de España y de la UE27
es = pd.read_csv(RAW / "comext_es_import_mensual.csv", dtype=str, keep_default_na=False)
ue = pd.read_csv(RAW / "comext_eu_import_mensual.csv", dtype=str, keep_default_na=False)
# Datacomex (API): importaciones de España por país de origen
dc = pd.read_csv(RAW / "datacomex_api_import_origen_mensual.csv", dtype=str, keep_default_na=False)

for nombre, d in {"ES import": es, "UE import": ue, "Datacomex": dc}.items():
    print(f"{nombre}: {d.shape[0]:,} filas × {d.shape[1]} columnas")

ES import: 71,289 filas × 8 columnas
UE import: 136,007 filas × 8 columnas
Datacomex: 56,947 filas × 10 columnas


## ✅Tarea 1. Separar los agregados de socios (Comext)
**Enunciado:** Separar en una tabla aparte las filas de agregados (`WORLD`, `EXT_*`, `INT_*`), que suman a otros socios y harían contar el comercio dos veces. No se elimina ninguna fila: los agregados se guardan para validar los totales.

In [3]:
# Agregados: no son países, son sumas de otros socios
agregados = {"WORLD", "EXT_EU", "EXT_EU27_2020", "INT_EU", "INT_EU27_2020",
             "EXT_EA", "EXT_EA21", "INT_EA", "INT_EA21"}

es_agregados = es[es["partner"].isin(agregados)].copy()
es_socios = es[~es["partner"].isin(agregados)].copy()
ue_agregados = ue[ue["partner"].isin(agregados)].copy()
ue_socios = ue[~ue["partner"].isin(agregados)].copy()

# Comprobación: ninguna fila se pierde (socios + agregados = tabla original)
for nombre, original, socios, agr in [("ES", es, es_socios, es_agregados), ("UE", ue, ue_socios, ue_agregados)]:
    print(f"{nombre}: {len(socios):,} socios + {len(agr):,} agregados = {len(socios) + len(agr):,} "
          f"(original {len(original):,}) → {'OK' if len(socios) + len(agr) == len(original) else 'ERROR'}")

ES: 52,893 socios + 18,396 agregados = 71,289 (original 71,289) → OK
UE: 131,936 socios + 4,071 agregados = 136,007 (original 136,007) → OK


**📌Conclusión:**
- Los agregados se han separado en tablas aparte (`es_agregados`, `ue_agregados`), que se usarán solo para validar totales.
- No se ha perdido ninguna fila: España 52,893 filas de socios + 18,396 de agregados = 71,289 (original); UE 131,936 + 4,071 = 136,007 (original).
- A partir de aquí se trabaja con `es_socios` y `ue_socios`, para no contar el comercio dos veces.

## ✅Tarea 2. Clasificar los socios (Comext)
**Enunciado:** Añadir una columna `tipo_socio` que distinga país, territorio (Ceuta y Melilla) y no especificado (`QP`, `QQ`–`QZ`). Los no especificados se mantienen en la tabla, pero se excluirán del HHI.

In [4]:
# Clasificación de los socios reales (los agregados ya están separados)
no_especificados = {"QP", "QQ", "QR", "QS", "QU", "QV", "QW", "QX", "QY", "QZ"}   # QA (Qatar) es un país
territorios = {"XC", "XL"}                                                       # Ceuta y Melilla

def tipo_socio(codigo):
    if codigo in no_especificados:
        return "no especificado"
    if codigo in territorios:
        return "territorio"
    return "país"

for d in (es_socios, ue_socios):
    d["tipo_socio"] = d["partner"].map(tipo_socio)

print("ES:", es_socios["tipo_socio"].value_counts().to_dict())
print("UE:", ue_socios["tipo_socio"].value_counts().to_dict())

ES: {'país': 52792, 'territorio': 85, 'no especificado': 16}
UE: {'país': 128889, 'no especificado': 2952, 'territorio': 95}


**📌Conclusión:**
- Cada fila tiene ahora su `tipo_socio`: país, territorio (Ceuta y Melilla) o no especificado.
- España: 52,792 filas de país, 85 de territorio y 16 de no especificado. UE: 128,889 de país, 95 de territorio y 2,952 de no especificado.
- Los no especificados se mantienen en la tabla y en los totales, pero se excluirán del cálculo del HHI, porque no son un proveedor identificable. Ceuta y Melilla se tratan como socios (origen real fuera del territorio aduanero de la UE).

## ✅Tarea 3. Eliminar la columna `SUPPLEMENTARY_QUANTITY` (Comext)
**Enunciado:** Quitar la columna de unidades suplementarias, vacía en el 100 % de las filas (paso 1 del EDA). Se elimina una columna sin datos, no filas.

In [5]:
# Comprobación antes de quitarla: la columna debe estar vacía en todas las filas
for nombre, d in {"ES socios": es_socios, "ES agregados": es_agregados,
                  "UE socios": ue_socios, "UE agregados": ue_agregados}.items():
    print(f"{nombre}: filas con dato en SUPPLEMENTARY_QUANTITY = {(d['SUPPLEMENTARY_QUANTITY'] != '').sum()}")

es_socios = es_socios.drop(columns="SUPPLEMENTARY_QUANTITY")
es_agregados = es_agregados.drop(columns="SUPPLEMENTARY_QUANTITY")
ue_socios = ue_socios.drop(columns="SUPPLEMENTARY_QUANTITY")
ue_agregados = ue_agregados.drop(columns="SUPPLEMENTARY_QUANTITY")
print(es_socios.columns.tolist())

ES socios: filas con dato en SUPPLEMENTARY_QUANTITY = 0
ES agregados: filas con dato en SUPPLEMENTARY_QUANTITY = 0
UE socios: filas con dato en SUPPLEMENTARY_QUANTITY = 0
UE agregados: filas con dato en SUPPLEMENTARY_QUANTITY = 0
['periodo', 'reporter', 'partner', 'product', 'flow', 'VALUE_IN_EUROS', 'QUANTITY_IN_100KG', 'tipo_socio']


**📌Conclusión:** `SUPPLEMENTARY_QUANTITY` estaba vacía en las cuatro tablas (0 filas con dato), así que se elimina la columna sin perder información. No hay unidades (piezas): el kg es la única medida de cantidad.

## ✅Tarea 4. Tipos y unidades (Comext)
**Enunciado:** Pasar el valor a número decimal (`valor_eur`, euros), el peso a kilos (`peso_kg` = `QUANTITY_IN_100KG` × 100, porque la fuente lo da en cientos de kg) y el periodo a fecha (`fecha`, primer día del mes). Las columnas originales se mantienen para poder comprobar la conversión.

In [6]:
# Conversión de tipos en las cuatro tablas de Comext
for d in (es_socios, es_agregados, ue_socios, ue_agregados):
    d["valor_eur"] = pd.to_numeric(d["VALUE_IN_EUROS"]).astype(float)
    d["peso_kg"] = pd.to_numeric(d["QUANTITY_IN_100KG"]) * 100
    d["fecha"] = pd.to_datetime(d["periodo"], format="%Y-%m")

# Comprobación: sin valores que no se hayan podido convertir y periodo completo
for nombre, d in {"ES socios": es_socios, "UE socios": ue_socios}.items():
    print(f"{nombre}: NaN valor {d['valor_eur'].isna().sum()} | NaN peso {d['peso_kg'].isna().sum()} | "
          f"{d['fecha'].min().date()} → {d['fecha'].max().date()} ({d['fecha'].nunique()} meses)")
es_socios.dtypes

ES socios: NaN valor 0 | NaN peso 0 | 2015-01-01 → 2026-08-01 (140 meses)
UE socios: NaN valor 0 | NaN peso 0 | 2015-01-01 → 2026-08-01 (140 meses)


periodo                         str
reporter                        str
partner                         str
product                         str
flow                            str
VALUE_IN_EUROS                  str
QUANTITY_IN_100KG               str
tipo_socio                      str
valor_eur                   float64
peso_kg                     float64
fecha                datetime64[us]
dtype: object

**Validación de la tarea 1:** con el valor ya en número, se comprueba que la suma de todos los socios de España coincide con el agregado `WORLD` mes a mes.

In [7]:
# Validación: la suma de los socios de España coincide con el agregado WORLD (por mes)
suma_socios = es_socios.groupby("fecha")["valor_eur"].sum()
world = es_agregados[es_agregados["partner"] == "WORLD"].groupby("fecha")["valor_eur"].sum()
diferencia = (suma_socios - world).abs()
print(f"Total socios: {suma_socios.sum():,.2f} € | WORLD: {world.sum():,.2f} € | "
      f"diferencia máxima en un mes: {diferencia.max():,.2f} €")

Total socios: 27,416,813,949.00 € | WORLD: 27,416,813,949.00 € | diferencia máxima en un mes: 0.00 €


**📌Conclusión:**
- `valor_eur` y `peso_kg` son numéricas (float) y `fecha` es una fecha; ningún valor ha quedado sin convertir (0 NaN). Las columnas originales se mantienen para poder comprobarlo.
- Periodo completo en España y en la UE: de enero de 2015 a agosto de 2026 (140 meses).
- Validación: la suma de los socios de España coincide con `WORLD` (27,416,813,949.00 €; diferencia 0.00 € en todos los meses). Separar los agregados no ha hecho perder ningún dato.

## ✅Tarea 5. Tipos y unidades (Datacomex)
**Enunciado:** Pasar `euros` y `kilos` (texto con coma decimal) a número como `valor_eur` y `peso_kg`, y `periodo` ("Enero de 2015") a fecha (`fecha`), con los mismos nombres que en Comext. Las columnas originales se mantienen.

In [8]:
meses = {"Enero": 1, "Febrero": 2, "Marzo": 3, "Abril": 4, "Mayo": 5, "Junio": 6, "Julio": 7,
         "Agosto": 8, "Septiembre": 9, "Octubre": 10, "Noviembre": 11, "Diciembre": 12}
dc["valor_eur"] = pd.to_numeric(dc["euros"].str.replace(",", ".", regex=False))
dc["peso_kg"] = pd.to_numeric(dc["kilos"].str.replace(",", ".", regex=False))
mes_anio = dc["periodo"].str.split(" de ", expand=True)
dc["fecha"] = pd.to_datetime(dict(year=mes_anio[1].astype(int), month=mes_anio[0].map(meses), day=1))

print(f"NaN valor {dc['valor_eur'].isna().sum()} | NaN peso {dc['peso_kg'].isna().sum()} | NaN fecha {dc['fecha'].isna().sum()}")
print(f"{dc['fecha'].min().date()} → {dc['fecha'].max().date()} ({dc['fecha'].nunique()} meses) | "
      f"total {dc['valor_eur'].sum():,.2f} €")

NaN valor 0 | NaN peso 0 | NaN fecha 0
2015-01-01 → 2026-08-01 (140 meses) | total 27,302,619,918.59 €


**📌Conclusión:**
- `valor_eur`, `peso_kg` y `fecha` creadas sin ningún valor perdido (0 NaN); periodo completo, de enero de 2015 a agosto de 2026 (140 meses).
- Total de Datacomex: 27,302,619,918.59 €, un 0.42 % menos que Comext (27,416,813,949.00 €). La diferencia está en los años con datos provisionales (2024–2026), que se revisan después de publicarse; hasta 2023 las dos fuentes coinciden casi al euro (paso 8 del EDA).

## ✅Tarea 6. Clasificar los países (Datacomex)
**Enunciado:** Añadir `tipo_socio`: los códigos 952 (avituallamiento) y 958–960 (países y territorios no determinados) son "no especificado" y se excluirán del HHI; el resto, "país". Se trabaja con `id_pais` porque la API corta los nombres largos.

In [9]:
no_especificados_dc = {"952", "958", "959", "960"}
dc["tipo_socio"] = np.where(dc["id_pais"].isin(no_especificados_dc), "no especificado", "país")

resumen = dc.groupby("tipo_socio").agg(paises=("id_pais", "nunique"), filas=("id_pais", "size"),
                                       valor_eur=("valor_eur", "sum"))
resumen["% valor"] = resumen["valor_eur"] / resumen["valor_eur"].sum() * 100
resumen

,paises,filas,valor_eur,% valor
tipo_socio,,,,
no especificado,4,1468,"58,523,428.70",0.21
país,196,55479,"27,244,096,489.89",99.79


**📌Conclusión:** 196 países reales (99.79 % del valor) y 4 códigos no especificados (952 avituallamiento; 958–960 no determinados), con 1,468 filas y el 0.21 % del valor. Se mantienen en la tabla y en los totales, pero se excluirán del HHI.

## ✅Tarea 7. Región de origen (Datacomex)
**Enunciado:** Añadir dos columnas para el contraste del peso de Asia (Mann-Whitney) y los KPI: `ue` (UE / terceros países, con la composición actual de 27 países en todo el periodo, como Comext EU27_2020) y `region` (UE / Asia / resto). Asia = códigos 600–799 de la nomenclatura de países (Oriente Próximo, sur y este de Asia), salvo Chipre, que es de la UE.

In [10]:
# UE27 (composición actual) con los códigos de país de Datacomex; España (011) no aparece porque es el declarante
UE27 = {"001", "003", "004", "005", "007", "008", "009", "010", "017", "018", "030", "032", "038",
        "046", "053", "054", "055", "060", "061", "063", "064", "066", "068", "091", "092", "600"}
codigo = dc["id_pais"].astype(int)

dc["ue"] = np.where(dc["id_pais"].isin(UE27), "UE", "terceros países")
dc["region"] = np.select(
    [dc["tipo_socio"] == "no especificado", dc["id_pais"].isin(UE27), codigo.between(600, 799)],
    ["no especificado", "UE", "Asia"],
    default="resto")

# Comprobación: países de cada región (que no se cuele ninguno donde no toca)
for r in ["UE", "Asia"]:
    print(r, sorted(dc.loc[dc["region"] == r, "pais"].unique()), "\n")
print(f"Países en la UE: {dc.loc[dc['region'] == 'UE', 'id_pais'].nunique()} (deben ser 26: la UE27 sin España)")

UE ['Alemania', 'Austria', 'Bulgaria', 'Bélgica', 'Chipre', 'Croacia', 'Dinamarca', 'Eslovaquia', 'Eslovenia', 'Estonia', 'Finlandia', 'Francia', 'Grecia', 'Hungría', 'Irlanda', 'Italia', 'Letonia', 'Lituania', 'Luxemburgo', 'Malta', 'Países Bajos', 'Polonia', 'Portugal', 'República Checa', 'Rumanía', 'Suecia'] 

Asia ['Afganistán', 'Arabia Saudí', 'Bahrein', 'Bangladesh', 'Brunei Darussalam', 'Bután', 'Camboya (Campuchea)', 'China', 'Cisjordania/Franja de', 'Corea del', 'Emiratos Árabes', 'Filipinas', 'Hong-Kong', 'India', 'Indonesia', 'Irak', 'Irán', 'Islas Maldivas', 'Israel', 'Japón', 'Jordania', 'Kuwait', 'Laos', 'Líbano', 'Macao', 'Malasia', 'Myanmar', 'Nepal', 'Omán', 'Pakistán', 'Qatar', 'Singapur', 'Siria', 'Sri Lanka', 'Tailandia', 'Taiwán', 'Timor-Leste', 'Vietnam', 'Yemen'] 

Países en la UE: 26 (deben ser 26: la UE27 sin España)


**📌Conclusión:**
- `ue` separa UE y terceros países con la composición actual de 26 socios (la UE27 sin España) en todo el periodo, igual que Comext EU27_2020: el Reino Unido cuenta como tercer país también antes del Brexit, para que la serie sea comparable.
- `region` clasifica cada país en UE / Asia / resto / no especificado. Comprobado: los 26 de la UE son los correctos (incluido Chipre) y Asia agrupa 39 países, de Oriente Próximo (Israel, Emiratos…) al este de Asia (China, Japón, Taiwán, Corea, Malasia…).
- Limitación: Cáucaso y Asia Central (Georgia, Kazajstán…) quedan en "resto" por su código; su peso en chips es mínimo.

## ✅Tarea 8. Bandera de dato provisional (Datacomex)
**Enunciado:** Crear `dato_provisional` (True/False) a partir de la columna `mensaje`. Las filas provisionales se mantienen; la bandera sirve para mencionarlo en las limitaciones y, si hace falta, comprobar los resultados sin ellas.

In [11]:
dc["dato_provisional"] = dc["mensaje"] == "dato provisional"
print(dc.groupby(dc["fecha"].dt.year)["dato_provisional"].mean().mul(100).round(2).to_dict())

{2015: 0.0, 2016: 0.0, 2017: 0.0, 2018: 0.0, 2019: 0.0, 2020: 0.0, 2021: 0.0, 2022: 0.0, 2023: 0.0, 2024: 100.0, 2025: 100.0, 2026: 100.0}


**📌Conclusión:** son provisionales todos los datos de 2024, 2025 y 2026 (100 % de las filas) y ninguno hasta 2023. Se mantienen; la bandera permite mencionarlo en las limitaciones y comprobar si algún resultado cambia sin esos años.

## ✅Tarea 9. Armonizar los códigos de 2022 y crear la columna `bloque` (Comext y Datacomex)
**Enunciado:** Con la revisión del Sistema Armonizado de 2022, 854140 se dividió en 854141, 854142, 854143 y 854149, y 854150 en 854151 y 854159. Se crea la columna `familia`, que agrupa cada código nuevo con su código antiguo para tener series continuas de 2015 a 2026, y la columna `bloque`: `optoelectronica_fv` (familia 854140: fotovoltaica, LED y otros fotosensibles) o `chips` (el resto). El código original se mantiene.

In [12]:
# Familia armonizada: cada código nuevo de 2022 → su código antiguo; el resto se queda igual
FAMILIA_2022 = {"854141": "854140", "854142": "854140", "854143": "854140", "854149": "854140",
                "854151": "854150", "854159": "854150"}

def armonizar(d, col_codigo):
    """Añade 'familia' y 'bloque' a partir de la columna de código de cada fuente."""
    d["familia"] = d[col_codigo].map(FAMILIA_2022).fillna(d[col_codigo])
    d["bloque"] = np.where(d["familia"] == "854140", "optoelectronica_fv", "chips")

# Comext usa la columna 'product'; Datacomex, 'taric'
for d in (es_socios, es_agregados, ue_socios, ue_agregados):
    armonizar(d, "product")
armonizar(dc, "taric")

# Comprobación 1: qué códigos entran en cada familia (deben salir 13 familias)
print("Familias en Datacomex:", dc["familia"].nunique())
print(dc.groupby("familia")["taric"].unique()[["854140", "854150"]], "\n")

# Comprobación 2: reparto por bloque (% del valor y de los kg)
for nombre, d in {"Comext ES": es_socios, "Datacomex": dc}.items():
    b = d.groupby("bloque")[["valor_eur", "peso_kg"]].sum()
    print(nombre, "\n", (b / b.sum() * 100).round(2), "\n")

Familias en Datacomex: 13
familia
854140    [854140, 854141, 854142, 854143, 854149]
854150                    [854150, 854151, 854159]
Name: taric, dtype: object 

Comext ES 
                     valor_eur  peso_kg
bloque                                
chips                   43.68     5.95
optoelectronica_fv      56.32    94.05 

Datacomex 
                     valor_eur  peso_kg
bloque                                
chips                   43.88     5.19
optoelectronica_fv      56.12    94.81 



In [13]:
# Comprobación 3: continuidad de las familias en el cambio de códigos (Datacomex, M€ por mes).
# Si la armonización funciona, enero de 2022 no debe dar un salto respecto a los meses anteriores.
alrededor = dc[(dc["fecha"] >= "2021-10-01") & (dc["fecha"] <= "2022-03-01") & dc["familia"].isin(["854140", "854150"])]
(alrededor.groupby(["fecha", "familia"])["valor_eur"].sum() / 1e6).unstack("familia")

familia,854140,854150
fecha,,
2021-10-01,150.18,1.60
2021-11-01,197.25,0.88
2021-12-01,159.60,3.65
2022-01-01,131.04,1.27
2022-02-01,141.99,0.68
2022-03-01,199.29,1.88


**📌Conclusión:**
- Columna `familia` creada en Comext (España y UE, socios y agregados) y en Datacomex: 13 familias; los códigos de 2022 se agrupan con su código antiguo y el código original se mantiene.
- Continuidad comprobada: en enero de 2022 la familia 854140 no da ningún salto (131.04 M€, dentro del rango de los meses anteriores). La armonización funciona.
- Columna `bloque`: los chips son el 43.68 % del valor y el 5.95 % de los kg en Comext (43.88 % y 5.19 % en Datacomex). La fotovoltaica y LED se analizan aparte, solo como contexto.
- Datacomex tiene menos kg de chips (5.19 % frente a 5.95 %): coherente con que no incluye la fila con peso anómalo de Comext de 2024-12 (se trata en la tarea 10).

## ✅Tarea 10. Bandera de peso anómalo (Comext y Datacomex)
**Enunciado:** Marcar con `peso_anomalo` las filas de chips cuyo peso parece mal declarado. Una fila se marca solo si cumple **los dos criterios** a la vez:
1. **€/kg imposible:** más de 100,000 kg y un €/kg por debajo del 1 % del €/kg mediano de su código.
2. **Peso fuera de su serie:** más de 100 veces el peso mediano de su propia serie (país × código).

No se elimina ninguna fila ni se imputa el peso: la fila se mantiene en la tabla y en los totales de valor, y solo su peso no se suma en la serie de kg (`peso_kg_serie`). La fotovoltaica no se marca (no es objeto de la previsión).

In [14]:
# Umbrales del criterio (definidos en el paso 4 del EDA)
KG_MIN = 100_000          # criterio 1: solo envíos grandes
FACTOR_EUR_KG = 0.01      # criterio 1: €/kg < 1 % del €/kg mediano del código
FACTOR_SERIE = 100        # criterio 2: peso > 100 veces el mediano de su serie país × código

def marcar_peso_anomalo(d, col_pais, col_codigo):
    """Añade 'peso_anomalo' y 'peso_kg_serie'. Solo se evalúan las filas de chips."""
    chips = d["bloque"] == "chips"
    validas = chips & (d["peso_kg"] > 0) & (d["valor_eur"] > 0)
    eur_kg = (d["valor_eur"] / d["peso_kg"]).where(validas)                       # €/kg solo con filas válidas
    eur_kg_mediano_codigo = eur_kg.groupby(d[col_codigo]).transform("median")
    peso_mediano_serie = d["peso_kg"].where(validas).groupby([d[col_pais], d[col_codigo]]).transform("median")

    criterio_1 = (d["peso_kg"] > KG_MIN) & (eur_kg < FACTOR_EUR_KG * eur_kg_mediano_codigo)
    criterio_2 = d["peso_kg"] > FACTOR_SERIE * peso_mediano_serie
    d["peso_anomalo"] = chips & criterio_1 & criterio_2
    d["peso_kg_serie"] = d["peso_kg"].where(~d["peso_anomalo"])                   # NaN en las filas marcadas
    return criterio_1 & chips

for nombre, d, col_pais, col_codigo in [("Comext ES", es_socios, "partner", "product"),
                                        ("Comext UE", ue_socios, "partner", "product"),
                                        ("Datacomex", dc, "id_pais", "taric")]:
    solo_criterio_1 = marcar_peso_anomalo(d, col_pais, col_codigo)
    chips = d[d["bloque"] == "chips"]
    marcadas = chips[chips["peso_anomalo"]]
    print(f"{nombre}: criterio 1 → {solo_criterio_1.sum()} filas | los dos criterios → {len(marcadas)} filas | "
          f"{marcadas['peso_kg'].sum() / chips['peso_kg'].sum() * 100:.2f} % de los kg de chips | "
          f"{marcadas['valor_eur'].sum() / chips['valor_eur'].sum() * 100:.3f} % del valor de chips")

Comext ES: criterio 1 → 17 filas | los dos criterios → 9 filas | 14.49 % de los kg de chips | 0.079 % del valor de chips
Comext UE: criterio 1 → 152 filas | los dos criterios → 66 filas | 8.58 % de los kg de chips | 0.039 % del valor de chips
Datacomex: criterio 1 → 37 filas | los dos criterios → 11 filas | 4.57 % de los kg de chips | 0.072 % del valor de chips


In [15]:
# Filas marcadas: revisar una a una que son errores de peso y no envíos reales
cols = ["fecha", "valor_eur", "peso_kg"]
print("Comext ES")
display(es_socios.loc[es_socios["peso_anomalo"], ["partner", "product"] + cols].sort_values("peso_kg", ascending=False))
print("Datacomex")
display(dc.loc[dc["peso_anomalo"], ["pais", "taric"] + cols].sort_values("peso_kg", ascending=False))

Comext ES


,partner,product,fecha,valor_eur,peso_kg
16746,DE,854130,2024-12-01,"4,121,060.00","30,443,264.00"
64787,CZ,854239,2024-05-01,"957,306.00","4,062,373.00"
16121,DE,854130,2022-03-01,"2,016,299.00","2,259,900.00"
16298,NL,854130,2022-03-01,"521,414.00","878,057.00"
32624,NL,854159,2024-11-01,"593,069.00","751,521.00"
48866,DE,854232,2015-03-01,"898,349.00","550,615.00"
69361,IE,854290,2022-05-01,"294,500.00","294,500.00"
49992,IT,854232,2017-07-01,"8,407.00","245,097.00"
3199,HU,854110,2022-04-01,"63,009.00","147,175.00"


Datacomex


,pais,taric,fecha,valor_eur,peso_kg
52093,República Checa,854239,2024-05-01,"1,031,179.19","3,763,135.82"
13158,Alemania,854130,2022-03-01,"1,918,412.39","2,259,155.87"
13146,Países Bajos,854130,2022-03-01,"482,694.09","877,848.00"
13160,Alemania,854130,2022-05-01,"1,293,188.97","786,515.31"
25876,Países Bajos,854159,2024-11-01,"495,615.43","751,176.00"
25883,Alemania,854159,2024-06-01,"1,823,984.33","724,796.07"
38567,Alemania,854232,2015-03-01,"679,527.41","548,773.95"
52094,República Checa,854239,2024-06-01,"475,852.57","301,837.47"
55474,Irlanda,854290,2022-05-01,"294,500.00","294,500.00"
39453,Italia,854232,2017-07-01,"7,834.67","245,100.17"


**📌Conclusión:**
- Con los dos criterios se marcan 9 filas en Comext España (14.49 % de los kg de chips y 0.079 % del valor), 66 en Comext UE (8.58 % y 0.039 %) y 11 en Datacomex (4.57 % y 0.072 %). Con solo el primer criterio serían 17, 152 y 37: el segundo evita marcar envíos grandes pero normales para su serie.
- Son errores de peso, no de valor: el valor se mantiene en los totales y solo el peso deja de contar en la serie de kg (`peso_kg_serie`). No se elimina ninguna fila.
- El error más grande de Comext (Alemania, 854130, diciembre de 2024: 30.4 millones de kg, 0.14 €/kg) no existe en Datacomex, que además tiene el peso con precisión de gramos. Por eso la serie de kg de la previsión se toma de Datacomex (decisión del 11 nov).

## ✅Tarea 11. Banderas de peso 0 y valor 0 y €/kg (Comext y Datacomex)
**Enunciado:** Marcar las filas con peso 0 (`peso_0`: envíos de menos de 0.5 kg, que la fuente redondea a 0) o valor 0 (`valor_0`), y calcular el €/kg (`eur_kg`) solo con las filas válidas (`eur_kg_valido`); en el resto, el €/kg queda vacío (NaN). Todas las filas se mantienen en la tabla y en los totales de valor y de kg.

In [16]:
for nombre, d in {"Comext ES": es_socios, "Comext UE": ue_socios, "Datacomex": dc}.items():
    d["peso_0"] = d["peso_kg"] == 0
    d["valor_0"] = d["valor_eur"] == 0
    d["eur_kg_valido"] = ~d["peso_0"] & ~d["valor_0"]
    d["eur_kg"] = np.where(d["eur_kg_valido"], d["valor_eur"] / d["peso_kg"], np.nan)
    print(f"{nombre}: peso 0 → {d['peso_0'].sum():,} | valor 0 → {d['valor_0'].sum():,} | "
          f"sin €/kg válido → {(~d['eur_kg_valido']).sum():,} de {len(d):,} filas | "
          f"chips sin €/kg válido → {(~d['eur_kg_valido'] & (d['bloque'] == 'chips')).sum():,}")

# Comprobación: el €/kg no tiene infinitos ni ceros
for nombre, d in {"Comext ES": es_socios, "Datacomex": dc}.items():
    print(f"{nombre}: infinitos {np.isinf(d['eur_kg']).sum()} | €/kg = 0 → {(d['eur_kg'] == 0).sum()}")

Comext ES: peso 0 → 2,106 | valor 0 → 26 | sin €/kg válido → 2,128 de 52,893 filas | chips sin €/kg válido → 1,907
Comext UE: peso 0 → 8,538 | valor 0 → 45 | sin €/kg válido → 8,549 de 131,936 filas | chips sin €/kg válido → 7,687
Datacomex: peso 0 → 0 | valor 0 → 0 | sin €/kg válido → 0 de 56,947 filas | chips sin €/kg válido → 0
Comext ES: infinitos 0 | €/kg = 0 → 0
Datacomex: infinitos 0 | €/kg = 0 → 0


**📌Conclusión:**
- Comext España: 2,106 filas con peso 0 y 26 con valor 0 (4 con las dos cosas): 2,128 filas (4.0 %) sin €/kg válido, 1,907 de ellas de chips. En la UE, 8,549 filas.
- Datacomex: ninguna fila con peso 0 ni valor 0; todas tienen €/kg válido (el peso viene en gramos).
- Las filas se mantienen en la tabla y en los totales de valor y kg; solo se excluyen del €/kg, que queda sin infinitos ni ceros falsos. Su efecto en los totales es despreciable (< 0.01 % del valor).

## ✅Tarea 12. TIO 2022 (contexto)
**Enunciado:** Preparar la TIO 2022 para la Introducción y para elegir el sector de la empresa ficticia:
- quitar la columna vacía (`Unnamed: 0`) y usar la primera fila (números de rama) como **código de cada sector**, porque es parte del encabezado y no un dato;
- quedarse con el bloque de productos (64 productos + 44a + total) de la Tabla 3 (importado) y de la Tabla 1 (total); el resto de la hoja (VAB, empleo…) no se usa y queda en `raw`;
- crear una tabla de sectores (código, nombre y NACE, de la Tabla 8) y la tabla del producto 17, **CPA 26 "Productos informáticos, electrónicos y ópticos"**: importado, total y % importado por sector.

Recordatorio: CPA 26 es electrónica en general (incluye los chips, pero también ordenadores, móviles y óptica), en millones de €, de un solo año.

In [17]:
TIO = RAW / "cne_tio_22.xlsx"
tio_t3 = pd.read_excel(TIO, sheet_name="Tabla3", header=7, index_col=1)   # uso de lo importado
tio_t1 = pd.read_excel(TIO, sheet_name="Tabla1", header=7, index_col=1)   # uso total (nacional + importado)
tio_t8 = pd.read_excel(TIO, sheet_name="Tabla8", header=7)                # correspondencias rama ↔ NACE

def limpiar_tio(t):
    """Quita la columna vacía, separa la fila de códigos de rama y deja solo el bloque de productos."""
    t = t.drop(columns="Unnamed: 0")
    codigos = t.iloc[0]                       # fila de números de rama: encabezado, no dato
    t = t.iloc[1:]
    t = t[t.index.notna()]                    # filas en blanco de separación
    fin = list(t.index).index("Total de empleos a precios básicos")
    return t.iloc[:fin + 1], codigos          # 65 productos + fila de total

tio_importado, codigos_rama = limpiar_tio(tio_t3)
tio_total, _ = limpiar_tio(tio_t1)
print("Importado:", tio_importado.shape, "| Total:", tio_total.shape,
      "| mismas columnas:", (tio_importado.columns == tio_total.columns).all())

Importado: (66, 78) | Total: (66, 78) | mismas columnas: True


In [18]:
# Tabla de sectores (ramas): código, nombre y código NACE
tio_ramas = pd.DataFrame({"rama_nombre": codigos_rama.index, "rama_codigo": codigos_rama.values})
tio_ramas = tio_ramas[tio_ramas["rama_codigo"].notna()].copy()            # las columnas de demanda final no tienen código
tio_ramas["rama_codigo"] = tio_ramas["rama_codigo"].map(lambda x: x if isinstance(x, str) else str(int(x)))

t8 = tio_t8[tio_t8["Unnamed: 6"].notna()].copy()
t8["rama_codigo"] = t8["Unnamed: 6"].map(lambda x: x if isinstance(x, str) else str(int(x)))
tio_ramas = tio_ramas.merge(t8[["rama_codigo", "NACE rev. 2"]].rename(columns={"NACE rev. 2": "nace"}),
                            on="rama_codigo", how="left")
print(f"Sectores: {len(tio_ramas)} | sin NACE: {tio_ramas['nace'].isna().sum()}")
tio_ramas[tio_ramas["nace"].isna()]

Sectores: 65 | sin NACE: 1


,rama_nombre,rama_codigo,nace
44,Alquileres imputados de las viviendas ocupadas...,44a,NaN


In [19]:
# Producto 17 (CPA 26): cuánto usa cada sector de electrónica importada y qué % de lo que usa es importado
fila = "17. Productos informáticos, electrónicos y ópticos"
tio_cpa26 = tio_ramas.copy()
tio_cpa26["importado_m_eur"] = tio_cpa26["rama_nombre"].map(tio_importado.loc[fila]).astype(float)
tio_cpa26["total_m_eur"] = tio_cpa26["rama_nombre"].map(tio_total.loc[fila]).astype(float)
tio_cpa26["porcentaje_importado"] = np.where(tio_cpa26["total_m_eur"] > 0,
tio_cpa26["importado_m_eur"] / tio_cpa26["total_m_eur"] * 100, np.nan)

# Comprobación: la suma de los sectores coincide con el total de demanda intermedia de la TIO
print(f"Suma de sectores: {tio_cpa26['importado_m_eur'].sum():,.2f} M€ | "
      f"Total demanda intermedia: {tio_importado.loc[fila, 'Total demanda intermedia']:,.2f} M€")
tio_cpa26.sort_values("importado_m_eur", ascending=False).head(10)

Suma de sectores: 13,147.30 M€ | Total demanda intermedia: 13,147.30 M€


,rama_nombre,rama_codigo,nace,importado_m_eur,total_m_eur,porcentaje_importado
38,Servicios de telecomunicaciones,39,61,"1,764.80","2,394.20",73.71
39,"Servicios de programación, consultoría y otros...",40,62-63,"1,537.00","1,919.70",80.06
46,Servicios técnicos de arquitectura e ingenierí...,46,71,"1,079.90","1,303.10",82.87
59,"Servicios deportivos, recreativos y de entrete...",59,93,905.10,984.40,91.94
16,"Productos informáticos, electrónicos y ópticos",17,26,847.60,880.20,96.30
26,Construcciones y trabajos de construcción,27,41-43,635.70,774.50,82.08
22,Servicios de reparación e instalación de maqui...,23,33,547.60,577.40,94.84
48,Servicios de publicidad y de estudio de mercado,48,73,489.60,"1,007.90",48.58
53,Servicios de seguridad e investigación; servic...,53,80-82,474.20,543.80,87.20
17,Equipo eléctrico,18,27,470.00,481.60,97.59


**📌Conclusión:**
- TIO 2022 preparada: matrices de importado (Tabla 3) y total (Tabla 1) con 65 productos + total y las mismas 78 columnas (65 sectores + 13 de demanda final); la fila de números de rama se usa como código de sector.
- Tabla de sectores con código y NACE (65; solo el 44a, alquileres imputados, sin NACE, porque no es un sector productivo).
- Producto 17 (CPA 26): la suma por sectores cuadra con el total de demanda intermedia (13,147.30 M€).
- Los sectores que más electrónica importada usan son telecomunicaciones, servicios informáticos e ingeniería; la automoción no está entre los primeros. Se tendrá en cuenta al elegir el sector de la empresa ficticia (CPA 26 incluye mucho más que chips).

## ✅Modelo de datos (esquema en estrella)
**Enunciado:** Organizar las tablas limpias en un modelo relacional para el análisis y el dashboard: **tablas de hechos** (una fila por mes × país × código, con las medidas y las banderas) y **dimensiones** (los atributos de fechas, productos y países, una fila por clave). Se comprueba que no se pierde ninguna fila y que todas las claves de los hechos existen en sus dimensiones.

| Tabla | Tipo | Clave |
|---|---|---|
| `hechos_origen` | Hechos (Datacomex, España por país de origen) | fecha + id_pais + codigo |
| `hechos_expedicion_es` | Hechos (Comext, España por país de expedición) | fecha + partner + codigo |
| `hechos_expedicion_ue` | Hechos (Comext, UE27) | fecha + partner + codigo |
| `dim_fecha` | Dimensión | fecha |
| `dim_producto` | Dimensión | codigo |
| `dim_pais_origen` | Dimensión (Datacomex) | id_pais |
| `dim_socio_comext` | Dimensión (Comext) | partner |

Fuera del modelo: agregados de Comext (solo validación) y TIO 2022 (contexto, sin relación con las demás).

In [20]:
# --- dimensiones_fecha: un mes por fila ---
dim_fecha = pd.DataFrame({"fecha": pd.date_range(dc["fecha"].min(), dc["fecha"].max(), freq="MS")})
dim_fecha["anio"] = dim_fecha["fecha"].dt.year
dim_fecha["mes"] = dim_fecha["fecha"].dt.month
dim_fecha["trimestre"] = dim_fecha["fecha"].dt.quarter

# --- dimensiones_producto: los 19 códigos HS6 (descripción en inglés, de Comext) ---
prod = pd.read_csv(RAW / "comext_codigos_productos.csv", dtype=str, keep_default_na=False)
dim_producto = prod.copy()
dim_producto["partida"] = dim_producto["codigo"].str[:4]
dim_producto["familia"] = dim_producto["codigo"].map(FAMILIA_2022).fillna(dim_producto["codigo"])
dim_producto["bloque"] = np.where(dim_producto["familia"] == "854140", "optoelectronica_fv", "chips")

# --- dimensiones_pais_origen (Datacomex): un país por fila, con nombres completos donde la API los corta ---
cols_pais = ["pais", "tipo_socio", "ue", "region"]
assert (dc.groupby("id_pais")[cols_pais].nunique() == 1).all().all()   # cada código tiene un solo valor por atributo
dim_pais_origen = dc[["id_pais"] + cols_pais].drop_duplicates("id_pais").sort_values("id_pais").reset_index(drop=True)
nombres_completos = {"724": "Corea del Norte", "728": "Corea del Sur", "096": "Macedonia del Norte",
                     "045": "Ciudad del Vaticano", "952": "Avituallamiento terceros países",
                     "958": "Países y territorios no determinados (958)",
                     "959": "Países y territorios no determinados (959)",
                     "960": "Países y territorios no determinados (960)"}
dim_pais_origen["pais"] = dim_pais_origen["id_pais"].map(nombres_completos).fillna(dim_pais_origen["pais"])

# --- dimensiones_socio_comext: socios de España y de la UE, con nombre (inglés) y UE / terceros países ---
soc = pd.read_csv(RAW / "comext_nombres_socios_eu.csv", dtype=str, keep_default_na=False)
UE27_ISO = {"AT", "BE", "BG", "CY", "CZ", "DE", "DK", "EE", "ES", "FI", "FR", "GR", "HR", "HU",
            "IE", "IT", "LT", "LU", "LV", "MT", "NL", "PL", "PT", "RO", "SE", "SI", "SK"}
dim_socio_comext = (pd.concat([es_socios[["partner", "tipo_socio"]], ue_socios[["partner", "tipo_socio"]]])
                    .drop_duplicates("partner")
                    .merge(soc.rename(columns={"codigo": "partner"}), on="partner", how="left")
                    .sort_values("partner").reset_index(drop=True))
dim_socio_comext["ue"] = np.where(dim_socio_comext["partner"].isin(UE27_ISO), "UE", "terceros países")

for nombre, d in {"dim_fecha": dim_fecha, "dim_producto": dim_producto,
                  "dim_pais_origen": dim_pais_origen, "dim_socio_comext": dim_socio_comext}.items():
    print(f"{nombre}: {len(d)} filas | clave única: {d.iloc[:, 0].is_unique}")

dim_fecha: 140 filas | clave única: True
dim_producto: 19 filas | clave única: True
dim_pais_origen: 200 filas | clave única: True
dim_socio_comext: 244 filas | clave única: True


In [21]:
# Las tablas de hechos guardan solo las claves, las medidas y las banderas;
# los atributos (nombre del país, región, bloque…) están en las dimensiones.
medidas = ["valor_eur", "peso_kg", "peso_kg_serie", "eur_kg"]
banderas = ["peso_anomalo", "peso_0", "valor_0", "eur_kg_valido"]

hechos_origen = dc[["fecha", "id_pais", "taric"] + medidas + banderas + ["dato_provisional"]].rename(columns={"taric": "codigo"})
hechos_expedicion_es = es_socios[["fecha", "partner", "product"] + medidas + banderas].rename(columns={"product": "codigo"})
hechos_expedicion_ue = ue_socios[["fecha", "partner", "product"] + medidas + banderas].rename(columns={"product": "codigo"})

Celda de comprobación de integridad:

In [22]:
# 1) No se pierde ninguna fila ni ningún euro; 2) no hay claves duplicadas; 3) toda clave de los hechos está en su dimensión
pruebas = [("hechos_origen", hechos_origen, dc, "id_pais", dim_pais_origen),
           ("hechos_expedicion_es", hechos_expedicion_es, es_socios, "partner", dim_socio_comext),
           ("hechos_expedicion_ue", hechos_expedicion_ue, ue_socios, "partner", dim_socio_comext)]
for nombre, h, original, col_pais, dim_pais in pruebas:
    print(f"{nombre}: filas {len(h):,} (limpia {len(original):,}) | "
          f"valor igual: {np.isclose(h['valor_eur'].sum(), original['valor_eur'].sum())} | "
          f"duplicados: {h.duplicated(['fecha', col_pais, 'codigo']).sum()} | "
          f"fechas fuera de dim: {(~h['fecha'].isin(dim_fecha['fecha'])).sum()} | "
          f"países fuera de dim: {(~h[col_pais].isin(dim_pais[dim_pais.columns[0]])).sum()} | "
          f"códigos fuera de dim: {(~h['codigo'].isin(dim_producto['codigo'])).sum()}")

hechos_origen: filas 56,947 (limpia 56,947) | valor igual: True | duplicados: 0 | fechas fuera de dim: 0 | países fuera de dim: 0 | códigos fuera de dim: 0
hechos_expedicion_es: filas 52,893 (limpia 52,893) | valor igual: True | duplicados: 0 | fechas fuera de dim: 0 | países fuera de dim: 0 | códigos fuera de dim: 0
hechos_expedicion_ue: filas 131,936 (limpia 131,936) | valor igual: True | duplicados: 0 | fechas fuera de dim: 0 | países fuera de dim: 0 | códigos fuera de dim: 0


**📌Conclusión:**
- Modelo en estrella con 3 tablas de hechos (Datacomex por origen; Comext España y UE por expedición) y 4 dimensiones (fecha, producto, país de origen y socio de Comext).
- Integridad comprobada: los hechos tienen las mismas filas y el mismo valor que las tablas limpias, sin claves duplicadas, y todas sus claves existen en las dimensiones.
- En `dim_pais_origen` se completan los nombres que la API corta y que serían ambiguos (Corea del Norte / Corea del Sur, Macedonia del Norte, no determinados); otros nombres cortados menores (p. ej. "Trinidad y") se dejan como están.
- Las descripciones de producto y los nombres de socio de Comext están en inglés (fuente Eurostat).

## ✅Tarea 13. Guardar las tablas en `datos/procesados`
**Enunciado:** Guardar en CSV (UTF-8) las tablas del modelo (hechos y dimensiones), los agregados de Comext (solo para validar) y las tablas de la TIO. Después se vuelven a leer para comprobar que se guardan bien: mismas filas, mismos totales y códigos intactos (p. ej. `NA` = Namibia o `001` = Francia). Los archivos de `datos/raw` no se tocan.

In [23]:
# Agregados de Comext: solo para validar totales (mismas columnas que los hechos)
agregados_es = es_agregados[["fecha", "partner", "product", "valor_eur", "peso_kg"]].rename(columns={"product": "codigo"})
agregados_ue = ue_agregados[["fecha", "partner", "product", "valor_eur", "peso_kg"]].rename(columns={"product": "codigo"})

tablas = {
    # Modelo en estrella
    "hechos_origen": hechos_origen,
    "hechos_expedicion_es": hechos_expedicion_es,
    "hechos_expedicion_ue": hechos_expedicion_ue,
    "dim_fecha": dim_fecha,
    "dim_producto": dim_producto,
    "dim_pais_origen": dim_pais_origen,
    "dim_socio_comext": dim_socio_comext,
    # Validación y contexto
    "comext_agregados_es": agregados_es,
    "comext_agregados_ue": agregados_ue,
    "tio_ramas": tio_ramas,
    "tio_cpa26": tio_cpa26,
}
for nombre, t in tablas.items():
    t.to_csv(PROC / f"{nombre}.csv", index=False, encoding="utf-8")

# Matrices de la TIO: el índice (producto) se guarda como primera columna
tio_importado.to_csv(PROC / "tio_importado.csv", index_label="producto", encoding="utf-8")
tio_total.to_csv(PROC / "tio_total.csv", index_label="producto", encoding="utf-8")

for f in sorted(PROC.glob("*.csv")):
    print(f"{f.name}: {f.stat().st_size / 1e6:,.2f} MB")

comext_agregados_es.csv: 0.84 MB
comext_agregados_ue.csv: 0.22 MB
dim_fecha.csv: 0.00 MB
dim_pais_origen.csv: 0.01 MB
dim_producto.csv: 0.00 MB
dim_socio_comext.csv: 0.01 MB
hechos_expedicion_es.csv: 4.16 MB
hechos_expedicion_ue.csv: 10.79 MB
hechos_origen.csv: 5.03 MB
tio_cpa26.csv: 0.01 MB
tio_importado.csv: 0.03 MB
tio_ramas.csv: 0.00 MB
tio_total.csv: 0.04 MB


In [24]:
# Forma correcta de leer las tablas (la usaremos en el análisis y en Streamlit):
# - códigos como texto, para no perder ceros a la izquierda ("001");
# - keep_default_na=False + na_values=[""]: solo lo vacío es NaN, así "NA" (Namibia) se mantiene.
CODIGOS = {"id_pais": str, "partner": str, "codigo": str, "familia": str, "partida": str,
           "rama_codigo": str, "nace": str}

def leer(nombre):
    ruta = PROC / f"{nombre}.csv"
    columnas = pd.read_csv(ruta, nrows=0).columns
    return pd.read_csv(ruta, dtype=CODIGOS, keep_default_na=False, na_values=[""],
                       parse_dates=["fecha"] if "fecha" in columnas else False)

for nombre, original in tablas.items():
    leida = leer(nombre)
    mismo_valor = np.isclose(leida["valor_eur"].sum(), original["valor_eur"].sum()) if "valor_eur" in leida else "—"
    print(f"{nombre}: filas {len(leida):,} (original {len(original):,}) | columnas iguales: "
          f"{list(leida.columns) == list(original.columns)} | valor igual: {mismo_valor}")

print("\nNamibia en hechos_expedicion_es:", (leer("hechos_expedicion_es")["partner"] == "NA").sum(), "filas (deben ser 8)")
print("Francia en dim_pais_origen:", leer("dim_pais_origen").query("id_pais == '001'")["pais"].tolist())

hechos_origen: filas 56,947 (original 56,947) | columnas iguales: True | valor igual: True
hechos_expedicion_es: filas 52,893 (original 52,893) | columnas iguales: True | valor igual: True
hechos_expedicion_ue: filas 131,936 (original 131,936) | columnas iguales: True | valor igual: True
dim_fecha: filas 140 (original 140) | columnas iguales: True | valor igual: —
dim_producto: filas 19 (original 19) | columnas iguales: True | valor igual: —
dim_pais_origen: filas 200 (original 200) | columnas iguales: True | valor igual: —
dim_socio_comext: filas 244 (original 244) | columnas iguales: True | valor igual: —
comext_agregados_es: filas 18,396 (original 18,396) | columnas iguales: True | valor igual: True
comext_agregados_ue: filas 4,071 (original 4,071) | columnas iguales: True | valor igual: True
tio_ramas: filas 65 (original 65) | columnas iguales: True | valor igual: —
tio_cpa26: filas 65 (original 65) | columnas iguales: True | valor igual: —

Namibia en hechos_expedicion_es: 8 filas

**📌Conclusión:**
- Guardadas en `datos/procesados` (CSV, UTF-8) las 7 tablas del modelo en estrella, los agregados de Comext (solo validación) y las tablas de la TIO (sectores, CPA 26 y las dos matrices).
- Prueba de lectura correcta: mismas filas, mismas columnas y mismos totales que en memoria; los códigos se conservan como texto ("001" = Francia) y "NA" = Namibia no se pierde.
- Para leerlas en el análisis y en Streamlit se usa la función `leer()` (códigos como texto y solo lo vacío como NaN).
- `datos/raw` no se ha modificado.